# Pipeline Completo — AULA 04
## PDF → Markdown → 10 Estratégias de Chunking → Embeddings → JSON

### Fluxo
```
PDF → Markdown → Chunking (10 estratégias) → Embeddings → JSON
```

### Modelo de Embedding
| Opção | Modelo | Custo | Dimensão |
|-------|--------|-------|----------|
| HuggingFace ✅ | `all-MiniLM-L6-v2` | Gratuito | 384 |
| OpenRouter | `text-embedding-3-small` | Pago | 1536 |

### Estratégias
| # | Estratégia | Configuração |
|---|-----------|-------------|
| 1 | Fixo 200, sem overlap | tamanho extremo baixo |
| 2 | Fixo 500, sem overlap | tamanho padrão |
| 3 | Fixo 1000, sem overlap | bloco médio |
| 4 | Fixo 2000, sem overlap | tamanho extremo alto |
| 5 | Fixo 500, overlap 50 | overlap leve (10%) |
| 6 | Fixo 500, overlap 200 | overlap pesado (40%) |
| 7 | Por parágrafo | estrutura natural |
| 8 | Por sentença (3 agrupadas) | estrutura natural |
| 9 | Recursivo hierárquico | estratégia composta |
| 10 | Markdown H1/H2/H3 | estrutura semântica |

---
### 1. Instalação de Dependências

In [ ]:
!pip install -q pymupdf4llm langchain-text-splitters sentence-transformers numpy

---
### 2. Configuração

In [ ]:
import json
import re
import time
from pathlib import Path

import numpy as np
import pymupdf4llm
from google.colab import files
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)

# ─────────────────────────────────────────────────────────────────
# CONFIGURAÇÃO — modelo de embedding
#
#   USE_HUGGINGFACE = True  → gratuito, roda local no Colab
#   USE_HUGGINGFACE = False → OpenRouter API (requer créditos)
#                             configure OPENROUTER_API_KEY nos Secrets
# ─────────────────────────────────────────────────────────────────
USE_HUGGINGFACE = True

HF_MODEL   = 'sentence-transformers/all-MiniLM-L6-v2'
OR_MODEL   = 'openai/text-embedding-3-small'
BATCH_SIZE = 64
MAX_CHARS  = 6000

BASE_DIR = Path('/content/results')
BASE_DIR.mkdir(exist_ok=True)

# Inicializa modelo
if USE_HUGGINGFACE:
    print('Carregando modelo HuggingFace:', HF_MODEL)
    _hf_model = SentenceTransformer(HF_MODEL)
    EMBEDDING_MODEL = HF_MODEL
    EMBEDDING_DIM   = _hf_model.get_embedding_dimension()
    print(f'Modelo carregado! Dimensao: {EMBEDDING_DIM}')
else:
    from google.colab import userdata
    from openai import OpenAI
    _or_client = OpenAI(
        base_url='https://openrouter.ai/api/v1',
        api_key=userdata.get('OPENROUTER_API_KEY')
    )
    EMBEDDING_MODEL = OR_MODEL
    EMBEDDING_DIM   = 1536
    print('Usando OpenRouter:', OR_MODEL)

print('Configuracao concluida!')

---
### 3. Funções Auxiliares

In [ ]:
def limpar_texto(texto):
    texto = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', texto)
    linhas = [l for l in texto.splitlines() if len(l.strip()) > 4 or l.strip() == '']
    return '\n'.join(linhas)


def split_por_sentencas_agrupadas(texto, n=3):
    """Estrategia 8: divide em sentencas e agrupa N por chunk."""
    sentencas = re.split(r'(?<=[.!?])\s+', texto.strip())
    sentencas = [s.strip() for s in sentencas if len(s.strip()) > 10]
    chunks = []
    for i in range(0, len(sentencas), n):
        chunks.append(' '.join(sentencas[i:i + n]))
    return [c for c in chunks if len(c) > 10]


def gerar_embeddings(textos):
    """Gera embeddings com HuggingFace ou OpenRouter."""
    if USE_HUGGINGFACE:
        vetores = _hf_model.encode(
            textos,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        return vetores.tolist()
    else:
        vetores = []
        for i in range(0, len(textos), BATCH_SIZE):
            batch = [str(t)[:MAX_CHARS] for t in textos[i:i + BATCH_SIZE]]
            for tentativa in range(3):
                try:
                    resp = _or_client.embeddings.create(input=batch, model=OR_MODEL)
                    vetores.extend([d.embedding for d in resp.data])
                    break
                except Exception as e:
                    if tentativa < 2:
                        time.sleep(5)
                    else:
                        vetores.extend([[0.0] * EMBEDDING_DIM] * len(batch))
            if i + BATCH_SIZE < len(textos):
                time.sleep(0.5)
        return vetores


def dividir_chunks(texto, est):
    """Aplica a estrategia de chunking e retorna lista de dicts."""
    tipo = est['tipo']
    if tipo == 'sentenca':
        return [{'text': t, 'metadata': {}} for t in split_por_sentencas_agrupadas(texto)]
    if tipo == 'markdown':
        docs = est['splitter'].split_text(texto)
        return [{'text': d.page_content, 'metadata': dict(d.metadata)}
                for d in docs if d.page_content.strip()]
    textos = est['splitter'].split_text(texto)
    return [{'text': t, 'metadata': {}} for t in textos if t.strip() and len(t) >= 10]


print('Funcoes auxiliares prontas!')

---
### 4. Definição das 10 Estratégias

In [ ]:
ESTRATEGIAS = [
    {'test_id': 1,  'tipo': 'texto',    'strategy': 'fixed_200_no_overlap',
     'nome': 'Fixo 200, sem overlap',       'chunk_size': 200,  'chunk_overlap': 0,
     'splitter': CharacterTextSplitter(separator='', chunk_size=200,  chunk_overlap=0)},

    {'test_id': 2,  'tipo': 'texto',    'strategy': 'fixed_500_no_overlap',
     'nome': 'Fixo 500, sem overlap',       'chunk_size': 500,  'chunk_overlap': 0,
     'splitter': CharacterTextSplitter(separator='', chunk_size=500,  chunk_overlap=0)},

    {'test_id': 3,  'tipo': 'texto',    'strategy': 'fixed_1000_no_overlap',
     'nome': 'Fixo 1000, sem overlap',      'chunk_size': 1000, 'chunk_overlap': 0,
     'splitter': CharacterTextSplitter(separator='', chunk_size=1000, chunk_overlap=0)},

    {'test_id': 4,  'tipo': 'texto',    'strategy': 'fixed_2000_no_overlap',
     'nome': 'Fixo 2000, sem overlap',      'chunk_size': 2000, 'chunk_overlap': 0,
     'splitter': CharacterTextSplitter(separator='', chunk_size=2000, chunk_overlap=0)},

    {'test_id': 5,  'tipo': 'texto',    'strategy': 'fixed_500_overlap_50',
     'nome': 'Fixo 500, overlap 50',        'chunk_size': 500,  'chunk_overlap': 50,
     'splitter': CharacterTextSplitter(separator='', chunk_size=500,  chunk_overlap=50)},

    {'test_id': 6,  'tipo': 'texto',    'strategy': 'fixed_500_overlap_200',
     'nome': 'Fixo 500, overlap 200',       'chunk_size': 500,  'chunk_overlap': 200,
     'splitter': CharacterTextSplitter(separator='', chunk_size=500,  chunk_overlap=200)},

    {'test_id': 7,  'tipo': 'paragrafo','strategy': 'by_paragraph',
     'nome': 'Por paragrafo',               'chunk_size': 5000, 'chunk_overlap': 0,
     'splitter': CharacterTextSplitter(separator='\n\n', chunk_size=5000,
                                       chunk_overlap=0, is_separator_regex=False)},

    {'test_id': 8,  'tipo': 'sentenca', 'strategy': 'by_sentence_grouped_3',
     'nome': 'Por sentenca (3 agrupadas)',   'chunk_size': None, 'chunk_overlap': 0,
     'splitter': None},

    {'test_id': 9,  'tipo': 'texto',    'strategy': 'recursive_hierarchical',
     'nome': 'Recursivo hierarquico',        'chunk_size': 500,  'chunk_overlap': 50,
     'splitter': RecursiveCharacterTextSplitter(
         separators=['\n\n', '\n', '. ', '! ', '? ', ' ', ''],
         chunk_size=500, chunk_overlap=50)},

    {'test_id': 10, 'tipo': 'markdown', 'strategy': 'markdown_headers',
     'nome': 'Markdown H1/H2/H3',           'chunk_size': None, 'chunk_overlap': 0,
     'splitter': MarkdownHeaderTextSplitter(
         headers_to_split_on=[('#', 'h1'), ('##', 'h2'), ('###', 'h3')])},
]

print(f'{len(ESTRATEGIAS)} estrategias definidas!')

---
### 5. Upload e Conversão dos PDFs para Markdown

In [ ]:
print('Faca o upload dos PDFs:')
uploaded = files.upload()

documentos = {}  # doc_id → {nome, texto}

for nome_arquivo, conteudo in uploaded.items():
    print(f'\nConvertendo {nome_arquivo}...', end=' ')
    caminho_tmp = f'/tmp/{nome_arquivo}'
    with open(caminho_tmp, 'wb') as f:
        f.write(conteudo)
    try:
        md = pymupdf4llm.to_markdown(caminho_tmp)
        md = limpar_texto(md)
        titulo = nome_arquivo.replace('.pdf', '').replace('_', ' ').title()
        md = f'# {titulo}\n\n{md}'
        doc_id = re.sub(r'[^a-z0-9_]', '_', nome_arquivo.replace('.pdf', '').lower())

        # Salvar markdown
        pasta_md = BASE_DIR / doc_id / 'markdown'
        pasta_md.mkdir(parents=True, exist_ok=True)
        (pasta_md / f'{doc_id}.md').write_text(md, encoding='utf-8')

        documentos[doc_id] = {'nome': nome_arquivo, 'texto': md}
        print(f'OK ({len(md):,} chars)')
    except Exception as e:
        print(f'ERRO: {e}')

print(f'\nTotal: {len(documentos)} documentos convertidos.')

---
### 6. Pipeline: Chunking + Embeddings + JSON

In [ ]:
summary_docs = []

for doc_id, info in documentos.items():
    doc_nome = info['nome']
    texto    = info['texto']

    print(f'\n{"="*60}')
    print(f'Documento: {doc_nome} ({len(texto):,} chars)')
    print('='*60)

    experimentos = []

    for est in ESTRATEGIAS:
        test_id   = est['test_id']
        json_path = BASE_DIR / doc_id / f'test_{test_id:02d}' / 'chunks_embeddings.json'

        # Resume: pula se ja existe
        if json_path.exists():
            print(f'  Teste {test_id:02d}: ja existe, pulando.')
            with open(json_path, encoding='utf-8') as f:
                dados = json.load(f)
            sizes = [len(d['text']) for d in dados]
            experimentos.append({
                'test_id': test_id, 'strategy': est['strategy'],
                'chunk_size': est['chunk_size'], 'chunk_overlap': est['chunk_overlap'],
                'num_chunks': len(dados),
                'avg_chunk_size': round(float(np.mean(sizes)), 1) if sizes else 0,
                'min_chunk_size': min(sizes) if sizes else 0,
                'max_chunk_size': max(sizes) if sizes else 0,
                'embedding_model': EMBEDDING_MODEL,
                'embedding_dimension': len(dados[0]['embedding']) if dados else 0,
            })
            continue

        print(f'  Teste {test_id:02d}: {est["nome"]}')

        # Chunking
        try:
            chunk_items = dividir_chunks(texto, est)
            chunk_items = [c for c in chunk_items if len(c['text'].strip()) >= 10]
        except Exception as e:
            print(f'    ERRO ao dividir: {e}')
            continue

        print(f'    {len(chunk_items)} chunks gerados')

        # Embeddings
        textos_chunks = [c['text'] for c in chunk_items]
        vetores = gerar_embeddings(textos_chunks)

        # Montar JSON
        dados = []
        for idx, (chunk, vetor) in enumerate(zip(chunk_items, vetores), 1):
            dados.append({
                'chunk_id':      f'{doc_id}_test{test_id:02d}_chunk{idx:04d}',
                'document_id':   doc_id,
                'document_name': doc_nome,
                'test_id':       test_id,
                'strategy':      est['strategy'],
                'chunk_size':    est['chunk_size'],
                'chunk_overlap': est['chunk_overlap'],
                'text':          chunk['text'],
                'embedding':     vetor,
                'metadata': {
                    'char_count':      len(chunk['text']),
                    'embedding_model': EMBEDDING_MODEL,
                    **chunk['metadata'],
                },
            })

        # Salvar
        json_path.parent.mkdir(parents=True, exist_ok=True)
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(dados, f, ensure_ascii=False, indent=2)

        sizes = [d['metadata']['char_count'] for d in dados]
        experimentos.append({
            'test_id':             test_id,
            'strategy':            est['strategy'],
            'chunk_size':          est['chunk_size'],
            'chunk_overlap':       est['chunk_overlap'],
            'num_chunks':          len(dados),
            'avg_chunk_size':      round(float(np.mean(sizes)), 1),
            'min_chunk_size':      min(sizes),
            'max_chunk_size':      max(sizes),
            'embedding_model':     EMBEDDING_MODEL,
            'embedding_dimension': len(dados[0]['embedding']) if dados else 0,
        })

    summary_docs.append({
        'document_id':   doc_id,
        'document_name': doc_nome,
        'total_chars':   len(texto),
        'experiments':   experimentos,
    })

# summary.json
summary_path = BASE_DIR / 'summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump({'embedding_model': EMBEDDING_MODEL, 'documents': summary_docs},
              f, ensure_ascii=False, indent=2)

print(f'\nPipeline concluido! Summary em {summary_path}')

---
### 7. Resumo dos Experimentos

In [ ]:
import pandas as pd

for doc in summary_docs:
    print(f'\n{doc["document_name"]}')
    df = pd.DataFrame(doc['experiments'])[[
        'test_id', 'strategy', 'num_chunks', 'avg_chunk_size',
        'min_chunk_size', 'max_chunk_size'
    ]]
    print(df.to_string(index=False))

---
### 8. Download dos Resultados

In [ ]:
import shutil

print('Compactando resultados...')
shutil.make_archive('/content/resultados_aula04', 'zip', '/content/results')
print('Iniciando download...')
files.download('/content/resultados_aula04.zip')
print('Pronto! Salve o ZIP em AULA 04/results/')